# Analyze synthetic sales with parameterized SQL and exact money

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Build a small relational model, calculate line revenue and refunds, and verify aggregate SQL against an independent Python calculation.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library and pandas. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Create an in-memory database

Money is stored in integer cents and quantities in whole units. SQLite `STRICT` tables enforce these storage types; an ordinary `INTEGER` declaration alone would still allow fractional values. Strict mode permits lossless coercions, such as `2.0` to integer `2`, but rejects `2.5`. Quantities are positive for sales and negative for returns. Foreign keys and CHECK constraints reject unknown products, zero quantities and negative prices. An in-memory database disappears when its connection closes.

Strict tables require SQLite 3.37.0 or later. The collection's audited runtime meets that requirement; the version check below fails clearly on an older Python distribution.

SQLite stores signed 64-bit integers, and an overflowing multiplication can silently become a floating-point value even in a strict table. The final table constraint bounds each line product to the symmetric signed-integer range before it can enter a revenue calculation. An aggregate that exceeds SQLite's integer range raises an error rather than returning an exact total; production systems must choose and enforce an appropriate accounting range.

In [ ]:
import sqlite3

import pandas as pd

if sqlite3.sqlite_version_info < (3, 37, 0):
    raise RuntimeError("This tutorial requires SQLite 3.37.0 or later for STRICT tables")

connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")
connection.executescript("""
CREATE TABLE products (product_id TEXT PRIMARY KEY, category TEXT NOT NULL) STRICT;
CREATE TABLE sales (
    line_id TEXT PRIMARY KEY,
    product_id TEXT NOT NULL REFERENCES products(product_id),
    region TEXT NOT NULL,
    quantity INTEGER NOT NULL CHECK(quantity != 0),
    unit_price_cents INTEGER NOT NULL CHECK(unit_price_cents >= 0),
    CHECK (
        unit_price_cents = 0 OR quantity BETWEEN
            -(9223372036854775807 / unit_price_cents)
            AND 9223372036854775807 / unit_price_cents
    )
) STRICT;
""")
products = [("product-01", "hardware"), ("product-02", "supplies")]
sales = [
    ("line-01", "product-01", "region-A", 2, 1500),
    ("line-02", "product-02", "region-A", 3, 400),
    ("line-03", "product-01", "region-B", 1, 1500),
    ("line-04", "product-01", "region-A", -1, 1500),
]
connection.executemany("INSERT INTO products VALUES (?, ?)", products)
connection.executemany("INSERT INTO sales VALUES (?, ?, ?, ?, ?)", sales)
connection.commit()
pd.read_sql_query("SELECT * FROM sales ORDER BY line_id", connection)

## 2. Aggregate without a many-to-many join

The product primary key ensures each sales line joins to exactly one product. A nonunique dimension table would multiply rows and overstate revenue. Keep cents exact until the presentation layer.

In [ ]:
query = """
SELECT p.category, COUNT(*) AS line_count,
       SUM(s.quantity) AS net_units,
       SUM(s.quantity * s.unit_price_cents) AS net_revenue_cents
FROM sales AS s JOIN products AS p ON p.product_id = s.product_id
GROUP BY p.category ORDER BY p.category
"""
result = pd.read_sql_query(query, connection)
assert result.net_revenue_cents.sum() == 4200
assert result.set_index("category").loc["hardware", "net_units"] == 2


def format_cents(cents):
    # Integer division preserves low-order cents even near the 64-bit boundary.
    cents = int(cents)
    sign = "-" if cents < 0 else ""
    return f"{sign}{abs(cents) // 100}.{abs(cents) % 100:02d}"


assert format_cents(9223372036854775807) == "92233720368547758.07"
assert format_cents(-1) == "-0.01"
result.assign(net_revenue=result.net_revenue_cents.map(format_cents))

## 3. Bind user-controlled filter values

Only values are parameterized; SQL identifiers are fixed in code. A filter containing quote characters remains literal data and cannot become another SQL statement.

In [ ]:
def revenue_for_region(region):
    row = connection.execute(
        "SELECT COALESCE(SUM(quantity * unit_price_cents), 0) FROM sales WHERE region = ?",
        (region,),
    ).fetchone()
    return row[0]


assert revenue_for_region("region-A") == 2700
assert revenue_for_region("region-B") == 1500
assert revenue_for_region("unknown") == 0
assert revenue_for_region("' OR 1=1 --") == 0
print({region: revenue_for_region(region) for region in ["region-A", "region-B"]})

## 4. Reconcile totals and exercise constraints

An independent Python total checks the query. Invalid foreign keys, fractional quantities, fractional cents invalid signs and products outside the supported integer range must fail without changing the row count. Roll back each failed write before reusing the connection, then verify the remaining money and quantity fields have SQLite's integer storage class. The overflow case uses valid integer inputs whose product would otherwise become a floating-point value.

In [ ]:
expected_total = sum(quantity * price for _, _, _, quantity, price in sales)
assert expected_total == int(result.net_revenue_cents.sum())
invalid_rows = [
    ("invalid-product", "missing-product", "region-A", 1, 10),
    ("fractional-quantity", "product-01", "region-A", 1.5, 1500),
    ("fractional-cents", "product-01", "region-A", 1, 1500.5),
    ("zero-quantity", "product-01", "region-A", 0, 1500),
    ("negative-price", "product-01", "region-A", 1, -1),
    ("overflowing-product", "product-01", "region-A", 3, 2**62 + 1),
    ("overflowing-refund", "product-01", "region-A", -3, 2**62 + 1),
]
for invalid in invalid_rows:
    try:
        connection.execute("INSERT INTO sales VALUES (?, ?, ?, ?, ?)", invalid)
    except sqlite3.IntegrityError:
        connection.rollback()
    else:
        raise AssertionError(f"Invalid sales row accepted: {invalid[0]}")
assert connection.execute("SELECT COUNT(*) FROM sales").fetchone()[0] == 4
assert connection.execute(
    "SELECT DISTINCT typeof(quantity), typeof(unit_price_cents) FROM sales"
).fetchall() == [("integer", "integer")]
connection.close()
print("Reconciliation and seven constraint checks passed; database closed.")

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

This model assumes one currency, no taxes, no discounts and a single price per line. Production revenue accounting requires explicit rules for currencies, partial returns, recognition dates and canceled orders. The demonstration uses SQLite; review transaction and type semantics before porting to another database.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [Python sqlite3 parameter binding and transactions](https://docs.python.org/3/library/sqlite3.html)
- [SQLite STRICT table rules](https://www.sqlite.org/stricttables.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.